# 1. Library packages

In [1]:
setwd("/home/liyanguo/MyImmuCell/")

In [2]:
source("00_code_ref_atlas/0_ref_data_preperation/Environment.R")

# 2. R+python: Read, QC, celltypist, plot QC

In [ ]:
sample_list = list.files("01_rawdata/Reference_Atlas_scRNA")
length(sample_list)
sample_list

In [ ]:
if(T){
  sfInit(parallel = T,cpus = 24)
  sfLibrary(Seurat)
  sfLibrary(scCustomize)
  sfLibrary(anndata)
  sfLibrary(dplyr)
  sfLibrary(snowfall)
  sfLibrary(reticulate)
  sfLibrary(Azimuth)
  sfLapply(sample_list,function(i){
    print(paste0("Process sample ID: ", i),sep = "\n")

    #read scrna-seq raw data
    filedir = paste0("01_rawdata/Reference_Atlas_scRNA/",i,"/GeneFull_Ex50pAS/raw/")
    counts = Read10X(filedir)
    
    #read cellbender and check  
    cellbender = read.csv(paste0("01_rawdata/Reference_Atlas_scRNA/",i,"/outs/filtered/barcodes.tsv.gz"),header=F)
    counts = counts[,cellbender$V1]
      
    stopifnot("Check cellbender results!"= ncol(counts) == nrow(cellbender))

    object <- CreateSeuratObject(
      counts = counts,
      min.cells = 0,
      assay = "RNA",
      min.features = 0,
      project = i)

    if ("nFeature_RNA" %in% colnames(object@meta.data) & "nCount_RNA" %in% colnames(object@meta.data)) {
      print("exist")
    }else{
      object[["nCount_RNA"]] <- colSums(LayerData(object, "counts"))
      object[["nFeature_RNA"]] <- colSums(LayerData(object, "counts") > 0)
    }

    #Add SampleID and DonorID
    object = RenameCells(object = object, add.cell.id = i)
    object$SampleID = i
    object$DonorID = limma::strsplit2(object$SampleID,"_")[,1]
    #Add QC information
    object <- Add_Cell_QC_Metrics(object, species = "human",
                                  add_complexity = TRUE,
                                  add_top_pct = TRUE,
                                  add_IEG = TRUE,
                                  add_MSigDB = TRUE,
                                  add_cell_cycle = TRUE,
                                  add_mito_ribo = TRUE,
                                  add_hemo = TRUE,
                                  overwrite = TRUE)

    #celltypist
    source("00_code_ref_atlas/0_ref_data_preperation/ref_celltypist.R")
    pred = celltypist_prediction(object)
    object = AddMetaData(object,metadata = pred)
    
    #Pre Normalization and RunUMAP
    source("00_code_ref_atlas/0_ref_data_preperation/NormalizeData2Umap.R")
    object=NormalizeData2Umap(object)

    #Azimuth
    source("00_code_ref_atlas/0_ref_data_preperation/MyRunAzimuth.R")
    object <- MyRunAzimuth(object,
                       reference = "07_ref_model/Azimuth/pbmc_ref/",
                       verbose = F)
    
    #Doublet detection, doublets will removed during cell type annotation when cell with high level of expression of more than one cell population-specific markers (genes or proteins).
    source("00_code_ref_atlas/0_ref_data_preperation/Doublet_detection.R")
    object = Doublet_detection(object)

    #QC Figure: Doublet distribution, sample QC metrics, T/Myeloid/neutrophil markers.
    source("00_code_ref_atlas/0_ref_data_preperation/ref_single_donor_QC.R")
    single_donor_QC(object,"Ref")

    adata <- AnnData(
      X = t(LayerData(object, "counts")),
      obs = object@meta.data
    )
    
    write_h5ad(adata, paste0("02_Read_QC/Ref_RNA_h5ad/",i,".h5ad"),compression="gzip")
  })
  sfStop()
}

In [ ]:
# package version
sessionInfo()